# Hive Query Language

> **Before you start:** this lab runs on the Docker cluster's `hive` container (Apache Hive 4.1).
> `LOAD DATA LOCAL` reads files from *inside* that container, so copy the four CSV files from the `Lab` folder into it first (run from PowerShell inside the `Lab` folder):
>
> Run the cells in order. Lines starting with `--` are comments showing the general syntax.

In [ ]:
docker exec hive mkdir -p /home/user/lab
docker cp student.csv hive:/home/user/lab/
docker cp employees.csv hive:/home/user/lab/
docker cp employees_department.csv hive:/home/user/lab/
docker cp celestial_data.csv hive:/home/user/lab/

Open the Hive shell (Beeline) from the command line using the command below. Hive 4 has no old `hive` CLI anymore: Beeline is the shell, and it connects to HiveServer2 on port `10000`.
`-n user` logs in as the container's `user` account, which owns the Hive folders in HDFS. Without it you connect as `anonymous`, and `CREATE DATABASE` fails with *"Failed to create external path"*.

In [ ]:
docker exec -it hive beeline -u jdbc:hive2://localhost:10000 -n user --silent=true

> Tip: add `--silent=true` to the command above to hide the INFO log lines.

## HIVE QL DDL
* Create database:

In [ ]:
-- Syntax: CREATE DATABASE database_name;
CREATE DATABASE employee;

You can also use `IF NOT EXISTS` to avoid errors if the database already exists.

In [ ]:
CREATE DATABASE IF NOT EXISTS employee;

* Use The database created:
    Sets the current database context

In [ ]:
-- Syntax: USE database_name;
USE employee;

* Create table:
    Define a new table in Hive, specifying its schema, storage format, and location.

In [ ]:
-- Syntax: CREATE [EXTERNAL] TABLE [IF NOT EXISTS] table_name (column_name data_type, ...);
CREATE TABLE  employee(ID int, name string, salary float, Age int);

* Show database/ tables:
    List all the databases in the Hive metastore.

In [ ]:
SHOW DATABASES;

List all the tables in the current database.

In [ ]:
SHOW TABLES;

* Describe database/table:

In [ ]:
-- Syntax: DESCRIBE DATABASE [EXTENDED] database_name;
DESCRIBE DATABASE employee;

Shows the schema of a specified table, including column names and data types.

In [ ]:
-- Syntax: DESCRIBE [FORMATTED|EXTENDED] table_name;
DESCRIBE employee;

* Alter database:
    Used to modify the schema or properties of an existing table. Common uses include renaming tables or adding/dropping columns.

In [ ]:
-- Syntax: ALTER DATABASE database_name SET DBPROPERTIES ('key'='value');
ALTER DATABASE employee SET DBPROPERTIES ('owner'='depi');

-- Syntax: ALTER DATABASE Owner to user depi;
ALTER DATABASE employee SET OWNER USER depi;

In [ ]:
ALTER TABLE employee ADD COLUMNS (department STRING);

* Drop database/ table:
    We drop a temporary table and database here, so `employee` stays available for the rest of the lab.

In [ ]:
-- Syntax: DROP TABLE table_name;
CREATE TABLE temp_table (id INT);
DROP TABLE temp_table;

`CASCADE` is required when the database still contains tables.

In [ ]:
-- Syntax: DROP DATABASE database_name [CASCADE];
CREATE DATABASE temp_db;
DROP DATABASE temp_db CASCADE;

## Hive QL DML
* Load files into tables
    Load data from an HDFS file into a hive table (we use `LOAD DATA LOCAL` with real files later in the lab).

In [ ]:
-- Syntax: LOAD DATA [LOCAL] INPATH 'file_path' [OVERWRITE] INTO TABLE table_name;

* Inserting data into hive table from queries:

In [ ]:
-- Syntax: INSERT INTO TABLE table_name VALUES (value1, value2, ...);

INSERT INTO TABLE employee VALUES (1 ,'heba' ,1000, 34, 'hr');
INSERT INTO TABLE employee VALUES (2 ,'eman' ,1000, 34, 'sales');
INSERT INTO TABLE employee VALUES (3 ,'reem' ,3000, 45, 'sales');
INSERT INTO TABLE employee VALUES (4 ,'adel' ,4000, 40, 'marketing');
INSERT INTO TABLE employee VALUES (5 ,'rami' ,1000, 30, 'marketing');

*  To exit from the Hive shell (Beeline):  
    Use the command below. Don't run it now; we exit at the end of the lab.

In [ ]:
-- !quit

## Hive QL in Detail
* Create Internal Table
    `skip.header.line.count` skips the CSV header row when this table is read.

In [ ]:
CREATE TABLE employee1 (
id INT,
name STRING,
salary FLOAT,
age INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
TBLPROPERTIES ("skip.header.line.count"="1");

* Check Table Description

In [ ]:
DESCRIBE employee1;
DESCRIBE FORMATTED employee1;

    > **Hive 4 note:** `DESCRIBE FORMATTED` shows `Table Type: EXTERNAL_TABLE` with `external.table.purge = TRUE`, even though we didn't write `EXTERNAL`.
    > Since Hive 4, a "managed" table that isn't transactional (ACID) is stored as an external table with *purge* switched on, under `/user/hive/external/`.
    > It still behaves like a managed table: `DROP TABLE` deletes the data too. A real external table (next step) has no `purge` property, so `DROP TABLE` keeps the files.

* Create External Table

In [ ]:
CREATE EXTERNAL TABLE employee2 (
id INT,
name STRING,
salary FLOAT,
age INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ',';

* Create External Table with Location

In [ ]:
CREATE EXTERNAL TABLE employee_edu (
id INT,
name STRING,
salary FLOAT,
age INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION '/user/hive/external/employee_edu/';

* Alter Table Examples
    - Rename Table:

In [ ]:
ALTER TABLE employee2 RENAME TO emp_table;

- Add Column:

In [ ]:
ALTER TABLE emp_table ADD COLUMNS (surname STRING);

- Rename Column:

In [ ]:
ALTER TABLE emp_table CHANGE name first_name STRING;

* Create Partitioned Table (Static)
    - Create a new db to use for this part and call it students.

In [ ]:
CREATE DATABASE students;
USE students;

- Create a staging table and load the raw CSV into it. `student.csv` has all four columns (`id, name, course, age`), so this table is the source for the partition and bucket examples.

In [ ]:
CREATE TABLE edu_stud (
id INT,
name STRING,
course STRING,
age INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
TBLPROPERTIES ("skip.header.line.count"="1");

In [ ]:
LOAD DATA LOCAL INPATH '/home/user/lab/student.csv' INTO TABLE edu_stud;

- Create the partitioned table. The partition column `course` is not part of the column list.

In [ ]:
CREATE TABLE student (
id INT,
name STRING,
age INT
)
PARTITIONED BY (course STRING)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ',';

- Load Static Partitioned Data (we write the partition value ourselves):

In [ ]:
INSERT INTO TABLE student
PARTITION (course='Hadoop')
SELECT id, name, age FROM edu_stud WHERE course = 'Hadoop';

- Add more data to the next partition

In [ ]:
INSERT INTO TABLE student
PARTITION (course='Java')
SELECT id, name, age FROM edu_stud WHERE course = 'Java';

In [ ]:
INSERT INTO TABLE student
PARTITION (course='Python')
SELECT id, name, age FROM edu_stud WHERE course = 'Python';

In [ ]:
SHOW PARTITIONS student;

    > `LOAD DATA LOCAL INPATH '...' INTO TABLE student PARTITION (course='Hadoop');` also works for static partitions, but only when the file holds that course's rows alone and has no `course` column.

* Enable Dynamic Partitioning

In [ ]:
SET hive.exec.dynamic.partition = true;
SET hive.exec.dynamic.partition.mode = nonstrict;

- Create Table for Dynamic Partitioning

In [ ]:
create table student_part(
ID INT,
Name STRING,
Age INT
)
PARTITIONED BY (course STRING)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ',';

- Insert Into Partitioned Table Dynamically (the partition column must be the **last** column in the `SELECT`):

In [ ]:
INSERT INTO TABLE student_part
PARTITION (course)
SELECT id, name, age, course
FROM edu_stud;

In [ ]:
SHOW PARTITIONS student_part;

* Bucketing
    - Enable Bucketing (only needed on old Hive 1.x; Hive 2+, including our Hive 4.1, always buckets, and the setting is harmless here):

In [ ]:
SET hive.enforce.bucketing = true;

- Create Bucketed Table:

In [ ]:
CREATE TABLE student2 (
ID INT,
Name STRING,
Age INT,
Course STRING
)
CLUSTERED BY (id) INTO 3 BUCKETS
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ',';

- Insert Into Bucketed Table:
`LOAD DATA` only copies the file and does not split it into buckets, so we fill the table with `INSERT ... SELECT`.

In [ ]:
INSERT OVERWRITE TABLE student2
SELECT id, name, age, course FROM edu_stud;

## Basic Querying and Functions
* Switch back to the `employee` database:

In [ ]:
USE employee;

* Basic Query Operations

In [ ]:
SELECT * FROM employee;

SELECT salary + 500 FROM employee;
SELECT salary - 100 FROM employee;

SELECT * FROM employee WHERE salary >= 2500;
SELECT * FROM employee WHERE salary < 2500;

* Functions in Hive

In [ ]:
SELECT ID, Name, Age, SQRT(salary) FROM employee;
SELECT ID, UPPER(Name), Age, Salary FROM employee;
SELECT ID, LOWER(Name), Age, Salary FROM employee;

SELECT MIN(salary) FROM employee;
SELECT MAX(salary) FROM employee;

* Group By

In [ ]:
SELECT department, SUM(salary)
FROM employee
GROUP BY department;

* Order By / Sort By

In [ ]:
SELECT * FROM employee ORDER BY salary DESC;
SELECT * FROM employee SORT BY salary DESC;

* Join Operations
    - Create the table department and load both CSV files. `department.id` is the employee's id.

In [ ]:
create table department (
ID INT,
Department STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
TBLPROPERTIES ("skip.header.line.count"="1");

LOAD DATA LOCAL INPATH '/home/user/lab/employees_department.csv'
INTO TABLE department;

LOAD DATA LOCAL INPATH '/home/user/lab/employees.csv'
INTO TABLE employee1;

- Inner Join:

In [ ]:
SELECT e.name, d.department
FROM employee1 e
JOIN department d
ON e.id = d.id;

- Left Outer Join:

In [ ]:
SELECT e.name, d.department
FROM employee1 e
LEFT OUTER JOIN department d
ON e.id = d.id;

- Right Outer Join:

In [ ]:
SELECT e.name, d.department
FROM employee1 e
RIGHT OUTER JOIN department d
ON e.id = d.id;

- Full Outer Join:

In [ ]:
SELECT e.name, d.department
FROM employee1 e
FULL OUTER JOIN department d
ON e.id = d.id;

## Hive Practical Example (Astronomy.db)
Reference: https://labex.io/tutorials/hadoop-cosmic-hive-integration-journey-288977

* Creating a Hive Table
    In this step, we will create a Hive table to store the astronomical data collected from various celestial bodies.

    Create a new database called `astronomy`:

In [ ]:
CREATE DATABASE astronomy;

* Use the `astronomy` database:

In [ ]:
USE astronomy;

* Create a new table called celestial_bodies with the following schema:

In [ ]:
CREATE TABLE celestial_bodies (
    id INT,
    name STRING,
    type STRING,
    distance DOUBLE,
    mass DOUBLE,
    radius DOUBLE
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
TBLPROPERTIES ("skip.header.line.count"="1");

    This table will store information about various celestial bodies, including their ID, name, type (e.g., star, planet, asteroid), distance from Earth, mass, and radius.

* Load some sample data into the `celestial_bodies` table from a local file:

In [ ]:
LOAD DATA LOCAL INPATH '/home/user/lab/celestial_data.csv' OVERWRITE INTO TABLE celestial_bodies;

**Querying the Hive Table:**

In this step, we will execute some SQL-like queries on the `celestial_bodies` table to analyze the astronomical data.

* Get the count of celestial bodies in the table:

In [ ]:
SELECT COUNT(*) FROM celestial_bodies;

* Find the celestial bodies with a mass greater than `1.0`:

In [ ]:
SELECT name, type, mass FROM celestial_bodies WHERE mass > 1.0;

* Get the average distance of planets from Earth:

In [ ]:
SELECT AVG(distance) FROM celestial_bodies WHERE type = 'Planet';

* Exit the Hive shell (Beeline):

In [ ]:
!quit

* View the results in the browser (there is no Hue in this Docker setup):
    - HiveServer2 web UI: http://localhost:10002 shows the queries you ran.
    - HDFS NameNode UI: http://localhost:19870 (or http://localhost:29870 if `master2` is the active NameNode). Open **Utilities → Browse the file system** → `/user/hive/external` to see the folders and files behind each database, table, partition and bucket (e.g. `students.db/student_part/course=Hadoop/` and the 3 bucket files in `students.db/student2/`).